In [ ]:
# 单元格1：导入必要的库和设置基础路径
import h5py
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import datetime
import glob

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data'
data_path = os.path.join(base_dir, 'DATA/TRAIN38.mat')
output_base_dir = os.path.join(base_dir, 'processed_data')

# 创建输出目录
os.makedirs(output_base_dir, exist_ok=True)

print(f"数据路径: {data_path}")
print(f"输出目录: {output_base_dir}")

In [ ]:
# 单元格2：加载数据并进行初步分析 - 保留完整的原始数据结构
# 加载TRAIN38.mat文件
f = h5py.File(data_path, 'r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

# 提取数据
data = arrays['data'].transpose()  # 转置以获取正确的形状
region = arrays['region'].transpose()  # 102列的标签，每列表示一个区域
prob_idx = arrays['prob_idx'].transpose()  # 每个体素对应的病人ID
age = arrays['age'].transpose() if 'age' in arrays else None  # 每个体素对应的年龄

# 数据基本信息
print(f"数据形状: {data.shape}")
print(f"标签形状: {region.shape}")
print(f"病人索引形状: {prob_idx.shape}")
if age is not None:
    print(f"年龄数据形状: {age.shape}")
else:
    print("未找到年龄数据")

# 分析唯一的病人ID
unique_prob_idx = np.unique(prob_idx)
print(f"唯一病人ID: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 分析标签情况
if region.ndim == 2:
    # 计算每个区域标签中有多少个体素
    label_counts = np.sum(region, axis=0)
    for i in range(region.shape[1]):
        if label_counts[i] > 0:
            print(f"区域 {i}: {label_counts[i]} 个体素")
    
    # 计算每个体素被分配到了几个区域
    region_per_voxel = np.sum(region, axis=1)
    unique_counts, count_freqs = np.unique(region_per_voxel, return_counts=True)
    for count, freq in zip(unique_counts, count_freqs):
        print(f"{count} 个区域标签的体素数量: {freq}")
else:
    print("标签不是二维的，无法分析区域分布")

# 分析每个病人的样本数量
for idx in unique_prob_idx:
    count = np.sum(prob_idx == idx)
    print(f"病人 {idx}: {count}个样本")

In [ ]:
# 单元格3：按照病人ID划分数据集，保留完整的数据结构
# 设置随机种子确保结果可重现
np.random.seed(42)

# 病人划分
test_patients = [38]  # 第38号病人直接进入测试集
remaining_patients = [i for i in range(1, 38)]  # 剩余37个病人

# 随机选择7个病人加入测试集
additional_test_patients = np.random.choice(remaining_patients, 7, replace=False)
test_patients.extend(additional_test_patients)

# 剩余的30个病人用于训练和验证
train_val_patients = [p for p in remaining_patients if p not in additional_test_patients]

print(f"测试集病人 ({len(test_patients)}个): {sorted(test_patients)}")
print(f"训练和验证集病人 ({len(train_val_patients)}个): {sorted(train_val_patients)}")

# 根据病人ID划分数据
test_indices = np.where(np.isin(prob_idx, test_patients))[0]
train_val_indices = np.where(np.isin(prob_idx, train_val_patients))[0]

# 提取测试集 - 完整保留所有原始数据结构
test_data = data[test_indices]
test_regions = region[test_indices]
test_prob_idx = prob_idx[test_indices]
test_age = age[test_indices] if age is not None else None

# 提取训练和验证集 - 完整保留所有原始数据结构
train_val_data = data[train_val_indices]
train_val_regions = region[train_val_indices]
train_val_prob_idx = prob_idx[train_val_indices]
train_val_age = age[train_val_indices] if age is not None else None

print(f"测试集样本数: {len(test_data)}")
print(f"训练和验证集样本数: {len(train_val_data)}")

# 验证一一对应关系
print("\n验证数据分割后的一一对应关系:")
print(f"测试集: 数据形状 {test_data.shape}, 标签形状 {test_regions.shape}, 病人ID形状 {test_prob_idx.shape}")
if test_age is not None:
    print(f"测试集年龄数据形状: {test_age.shape}")
print(f"训练和验证集: 数据形状 {train_val_data.shape}, 标签形状 {train_val_regions.shape}, 病人ID形状 {train_val_prob_idx.shape}")
if train_val_age is not None:
    print(f"训练和验证集年龄数据形状: {train_val_age.shape}")

# 保存病人分配信息
patient_allocation = {
    "test_patients": sorted(test_patients),
    "train_val_patients": sorted(train_val_patients)
}

# 释放原始完整数据的内存
del data, region, prob_idx, age, arrays

In [ ]:
# 单元格4：创建和保存StandardScaler
# 功能：使用所有训练和验证数据拟合StandardScaler并保存

# 使用所有训练和验证数据拟合StandardScaler
print(f"使用 {train_val_data.shape[0]} 个样本拟合Scaler...")
scaler = StandardScaler()
scaler.fit(train_val_data)

# 保存Scaler
scaler_path = os.path.join(output_base_dir, 'data_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Scaler已拟合并保存到: {scaler_path}")
print(f"Scaler均值形状: {scaler.mean_.shape}")
print(f"Scaler方差形状: {scaler.var_.shape}")

In [ ]:
# 单元格5：按区域标签分组训练和验证数据，保持完整的数据结构
# 功能：将训练和验证数据按照102个区域标签分组，并按6:2比例拆分

# 创建输出目录
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')

for directory in [train_dir, val_dir, test_dir]:
    os.makedirs(directory, exist_ok=True)

# 找出哪些区域标签有数据
active_regions = []
for i in range(train_val_regions.shape[1]):
    if np.sum(train_val_regions[:, i]) > 0:
        active_regions.append(i)

print(f"活跃的区域标签数: {len(active_regions)}")
print(f"活跃的区域标签: {active_regions}")

# 将训练和验证数据按6:2比例分割
train_ratio = 0.6
np.random.seed(42)  # 确保结果可重现

# 对每个活跃区域进行处理
for region_idx in tqdm(active_regions, desc="处理区域标签"):
    # 找出该区域的所有体素
    indices = np.where(train_val_regions[:, region_idx] == 1)[0]
    
    if len(indices) == 0:
        print(f"区域 {region_idx} 没有数据，跳过")
        continue
    
    # 提取该区域的所有数据
    region_data = train_val_data[indices]
    region_labels = train_val_regions[indices]
    region_prob_idx = train_val_prob_idx[indices]
    region_age = train_val_age[indices] if train_val_age is not None else None
    
    # 随机打乱索引
    shuffle_indices = np.random.permutation(len(indices))
    
    # 按照打乱的索引重排数据
    region_data = region_data[shuffle_indices]
    region_labels = region_labels[shuffle_indices]
    region_prob_idx = region_prob_idx[shuffle_indices]
    if region_age is not None:
        region_age = region_age[shuffle_indices]
    
    # 按6:2比例拆分
    train_size = int(len(region_data) * train_ratio)
    
    # 训练集
    train_data = region_data[:train_size]
    train_labels = region_labels[:train_size]
    train_prob_idx = region_prob_idx[:train_size]
    train_age = region_age[:train_size] if region_age is not None else None
    
    # 验证集
    val_data = region_data[train_size:]
    val_labels = region_labels[train_size:]
    val_prob_idx = region_prob_idx[train_size:]
    val_age = region_age[train_size:] if region_age is not None else None
    
    print(f"区域 {region_idx}: 总样本 {len(region_data)}，训练集 {len(train_data)}，验证集 {len(val_data)}")
    
    # 应用StandardScaler标准化
    train_data_scaled = scaler.transform(train_data)
    val_data_scaled = scaler.transform(val_data)
    
    # 保存训练集
    train_file = os.path.join(train_dir, f"region_{region_idx}.mat")
    with h5py.File(train_file, 'w') as f:
        f.create_dataset('data', data=train_data_scaled)
        f.create_dataset('region', data=train_labels)
        f.create_dataset('prob_idx', data=train_prob_idx)
        if train_age is not None:
            f.create_dataset('age', data=train_age)
    
    # 保存验证集
    val_file = os.path.join(val_dir, f"region_{region_idx}.mat")
    with h5py.File(val_file, 'w') as f:
        f.create_dataset('data', data=val_data_scaled)
        f.create_dataset('region', data=val_labels)
        f.create_dataset('prob_idx', data=val_prob_idx)
        if val_age is not None:
            f.create_dataset('age', data=val_age)

print("训练和验证集处理完成！")

In [ ]:
# 单元格6：处理测试集数据
# 功能：按区域标签处理测试集数据，保持完整的数据结构

# 找出测试集中哪些区域标签有数据
test_active_regions = []
for i in range(test_regions.shape[1]):
    if np.sum(test_regions[:, i]) > 0:
        test_active_regions.append(i)

print(f"测试集活跃的区域标签数: {len(test_active_regions)}")
print(f"测试集活跃的区域标签: {test_active_regions}")

# 对每个活跃区域进行处理
for region_idx in tqdm(test_active_regions, desc="处理测试集区域标签"):
    # 找出该区域的所有体素
    indices = np.where(test_regions[:, region_idx] == 1)[0]
    
    if len(indices) == 0:
        print(f"测试集区域 {region_idx} 没有数据，跳过")
        continue
    
    # 提取该区域的所有数据
    region_data = test_data[indices]
    region_labels = test_regions[indices]
    region_prob_idx = test_prob_idx[indices]
    region_age = test_age[indices] if test_age is not None else None
    
    # 应用StandardScaler标准化
    region_data_scaled = scaler.transform(region_data)
    
    # 保存测试集
    test_file = os.path.join(test_dir, f"region_{region_idx}.mat")
    with h5py.File(test_file, 'w') as f:
        f.create_dataset('data', data=region_data_scaled)
        f.create_dataset('region', data=region_labels)
        f.create_dataset('prob_idx', data=region_prob_idx)
        if region_age is not None:
            f.create_dataset('age', data=region_age)
    
    print(f"测试集区域 {region_idx}: {len(region_data)} 个样本")
    
    # 可选：按病人ID分别保存
    unique_patients = np.unique(region_prob_idx)
    for patient_id in unique_patients:
        patient_indices = np.where(region_prob_idx == patient_id)[0]
        
        if len(patient_indices) == 0:
            continue
            
        patient_data = region_data_scaled[patient_indices]
        patient_labels = region_labels[patient_indices]
        patient_prob_idx = region_prob_idx[patient_indices]
        patient_age = region_age[patient_indices] if region_age is not None else None
        
        # 保存单个病人的数据
        patient_file = os.path.join(test_dir, f"patient_{int(patient_id)}_region_{region_idx}.mat")
        with h5py.File(patient_file, 'w') as f:
            f.create_dataset('data', data=patient_data)
            f.create_dataset('region', data=patient_labels)
            f.create_dataset('prob_idx', data=patient_prob_idx)
            if patient_age is not None:
                f.create_dataset('age', data=patient_age)

print("测试集处理完成！")

In [ ]:
# 单元格7：创建数据集索引文件
# 功能：为每个数据集创建索引文件，记录区域信息

def create_dataset_index(directory):
    """为指定目录创建数据集索引文件"""
    index_file = os.path.join(directory, "region_index.txt")
    
    with open(index_file, 'w') as f:
        f.write("region_id,sample_count,file_name,has_age\n")
        
        # 获取所有区域文件
        region_files = [file for file in os.listdir(directory) 
                      if file.endswith('.mat') and file.startswith('region_')]
        
        for file in sorted(region_files, key=lambda x: int(x.split('_')[1].split('.')[0])):
            # 从文件名提取区域ID
            region_id = int(file.split('_')[1].split('.')[0])
            
            # 读取文件获取样本数
            with h5py.File(os.path.join(directory, file), 'r') as mat_file:
                sample_count = mat_file['data'].shape[0]
                has_age = 'age' in mat_file
            
            f.write(f"{region_id},{sample_count},{file},{has_age}\n")
    
    print(f"索引文件已创建: {index_file}")
    
    # 如果有按病人分类的文件，也为它们创建索引
    patient_files = [file for file in os.listdir(directory) 
                    if file.endswith('.mat') and file.startswith('patient_')]
    
    if patient_files:
        patient_index_file = os.path.join(directory, "patient_index.txt")
        
        with open(patient_index_file, 'w') as f:
            f.write("patient_id,region_id,sample_count,file_name,has_age\n")
            
            for file in sorted(patient_files, key=lambda x: (int(x.split('_')[1]), int(x.split('_')[3].split('.')[0]))):
                # 从文件名提取信息
                parts = file.split('_')
                patient_id = int(parts[1])
                region_id = int(parts[3].split('.')[0])
                
                # 读取文件获取样本数
                with h5py.File(os.path.join(directory, file), 'r') as mat_file:
                    sample_count = mat_file['data'].shape[0]
                    has_age = 'age' in mat_file
                
                f.write(f"{patient_id},{region_id},{sample_count},{file},{has_age}\n")
        
        print(f"病人索引文件已创建: {patient_index_file}")

# 为每个数据集创建索引
print("创建数据集索引文件...")
create_dataset_index(train_dir)
create_dataset_index(val_dir)
create_dataset_index(test_dir)

In [ ]:
# 单元格8：创建数据加载辅助函数
# 功能：提供加载处理后数据的辅助函数

data_loader_file = os.path.join(output_base_dir, "data_loader.py")

with open(data_loader_file, 'w') as f:
    f.write("""# 数据加载辅助函数
import numpy as np
import os
import h5py
import glob
import pickle

def load_scaler(base_dir):
    \"\"\"加载保存的StandardScaler\"\"\"
    scaler_path = os.path.join(base_dir, 'data_scaler.pkl')
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

def load_region_index(directory):
    \"\"\"加载区域索引文件\"\"\"
    index_file = os.path.join(directory, "region_index.txt")
    index_data = {}
    
    with open(index_file, 'r') as f:
        # 跳过标题行
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                region_id = int(parts[0])
                sample_count = int(parts[1])
                file_name = parts[2]
                has_age = parts[3].lower() == 'true'
                
                index_data[region_id] = {
                    'sample_count': sample_count,
                    'file_name': file_name,
                    'has_age': has_age
                }
    
    return index_data

def load_region_data(directory, region_id):
    \"\"\"
    加载指定区域的数据
    
    参数:
        directory: 数据目录路径
        region_id: 区域ID
    
    返回:
        包含data, region, prob_idx和可能的age的字典
    \"\"\"
    index_data = load_region_index(directory)
    
    if region_id not in index_data:
        print(f"区域 {region_id} 在目录 {directory} 中不存在")
        return None
    
    file_name = index_data[region_id]['file_name']
    file_path = os.path.join(directory, file_name)
    
    result = {}
    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            result[key] = np.array(f[key])
    
    return result

def load_all_regions(directory, region_ids=None):
    \"\"\"
    加载指定目录下的所有区域数据或指定区域数据
    
    参数:
        directory: 数据目录路径
        region_ids: 要加载的区域ID列表，如果为None则加载所有区域
    
    返回:
        包含所有区域数据的字典，键为区域ID
    \"\"\"
    index_data = load_region_index(directory)
    
    if region_ids is None:
        region_ids = list(index_data.keys())
    
    result = {}
    for region_id in region_ids:
        if region_id in index_data:
            region_data = load_region_data(directory, region_id)
            if region_data is not None:
                result[region_id] = region_data
    
    return result

def load_patient_data(directory, patient_id, region_id=None):
    \"\"\"
    加载指定病人的数据
    
    参数:
        directory: 数据目录路径
        patient_id: 病人ID
        region_id: 区域ID（可选，如果只想加载特定区域的数据）
    
    返回:
        包含该病人数据的字典，键为区域ID
    \"\"\"
    patient_index_file = os.path.join(directory, "patient_index.txt")
    
    if not os.path.exists(patient_index_file):
        print(f"病人索引文件在目录 {directory} 中不存在")
        return {}
    
    patient_data = {}
    
    with open(patient_index_file, 'r') as f:
        # 跳过标题行
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 4:
                curr_patient_id = int(parts[0])
                curr_region_id = int(parts[1])
                file_name = parts[3]
                
                if curr_patient_id == patient_id and (region_id is None or curr_region_id == region_id):
                    file_path = os.path.join(directory, file_name)
                    
                    region_data = {}
                    with h5py.File(file_path, 'r') as f:
                        for key in f.keys():
                            region_data[key] = np.array(f[key])
                    
                    patient_data[curr_region_id] = region_data
    
    return patient_data

def load_dataset(base_dir, split='train', region_ids=None):
    \"\"\"
    加载指定数据集
    
    参数:
        base_dir: 基础目录路径
        split: 数据集类型，'train', 'val'或'test'
        region_ids: 要加载的区域ID列表，如果为None则加载所有区域
    
    返回:
        包含指定数据集所有区域数据的字典
    \"\"\"
    directory = os.path.join(base_dir, split)
    return load_all_regions(directory, region_ids)

def concatenate_regions(regions_data):
    \"\"\"
    将多个区域的数据合并成一个大数组
    
    参数:
        regions_data: 由load_all_regions返回的区域数据字典
    
    返回:
        包含合并后数据的字典，键为'data', 'region', 'prob_idx'和可能的'age'
    \"\"\"
    all_data = []
    all_region = []
    all_prob_idx = []
    all_age = []
    has_age = False
    
    for region_id, region_data in regions_data.items():
        all_data.append(region_data['data'])
        all_region.append(region_data['region'])
        all_prob_idx.append(region_data['prob_idx'])
        
        if 'age' in region_data:
            all_age.append(region_data['age'])
            has_age = True
    
    result = {
        'data': np.vstack(all_data) if all_data else np.array([]),
        'region': np.vstack(all_region) if all_region else np.array([]),
        'prob_idx': np.vstack(all_prob_idx) if all_prob_idx else np.array([])
    }
    
    if has_age and all_age:
        result['age'] = np.vstack(all_age)
    
    return result

# 使用示例
if __name__ == "__main__":
    # 替换为实际路径
    base_dir = "processed_data"
    
    # 加载Scaler
    scaler = load_scaler(base_dir)
    print(f"Scaler已加载，特征数量: {len(scaler.mean_)}")
    
    # 加载训练集中的某个区域
    region_id = 0  # 替换为实际区域ID
    region_data = load_region_data(os.path.join(base_dir, 'train'), region_id)
    if region_data:
        print(f"区域 {region_id} 的训练数据:")
        for key, value in region_data.items():
            print(f"  - {key}: 形状 {value.shape}")
    
    # 加载测试集中特定病人的数据
    patient_id = 38  # 替换为实际病人ID
    patient_data = load_patient_data(os.path.join(base_dir, 'test'), patient_id)
    print(f"病人 {patient_id} 在测试集中的数据:")
    for region_id, data in patient_data.items():
        print(f"  - 区域 {region_id}: {data['data'].shape[0]} 个样本")
""")

print(f"数据加载辅助函数已保存到: {data_loader_file}")

In [ ]:
# 单元格9：验证处理结果
# 功能：验证处理后的数据集是否正确保存和索引

print("验证处理结果...")

# 验证输出目录结构
print("1. 验证目录结构:")
for directory in [train_dir, val_dir, test_dir]:
    mat_files = [f for f in os.listdir(directory) if f.endswith('.mat')]
    region_files = [f for f in mat_files if f.startswith('region_')]
    patient_files = [f for f in mat_files if f.startswith('patient_')]
    
    print(f"  - {os.path.basename(directory)}目录:")
    print(f"    * {len(region_files)} 个区域数据文件")
    print(f"    * {len(patient_files)} 个病人数据文件")

# 验证索引文件
print("\n2. 验证索引文件:")
for directory in [train_dir, val_dir, test_dir]:
    index_file = os.path.join(directory, "region_index.txt")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            headers = next(f).strip().split(',')
            line_count = sum(1 for _ in f)
        print(f"  - {os.path.basename(directory)}区域索引文件: {line_count} 个条目")
        print(f"    * 索引字段: {headers}")
    
    patient_index_file = os.path.join(directory, "patient_index.txt")
    if os.path.exists(patient_index_file):
        with open(patient_index_file, 'r') as f:
            headers = next(f).strip().split(',')
            line_count = sum(1 for _ in f)
        print(f"  - {os.path.basename(directory)}病人索引文件: {line_count} 个条目")
        print(f"    * 索引字段: {headers}")

# 验证数据文件内容
print("\n3. 验证数据文件内容:")
for directory in [train_dir, val_dir, test_dir]:
    region_files = [f for f in os.listdir(directory) if f.endswith('.mat') and f.startswith('region_')]
    
    if region_files:
        sample_file = os.path.join(directory, region_files[0])
        with h5py.File(sample_file, 'r') as f:
            keys = list(f.keys())
            data_shape = f['data'].shape if 'data' in f else None
            region_shape = f['region'].shape if 'region' in f else None
            prob_idx_shape = f['prob_idx'].shape if 'prob_idx' in f else None
            age_shape = f['age'].shape if 'age' in f else None
            
            print(f"  - {os.path.basename(directory)}样本文件 {os.path.basename(sample_file)}:")
            print(f"    * 包含键: {keys}")
            print(f"    * 数据形状: {data_shape}")
            print(f"    * 标签形状: {region_shape}")
            print(f"    * 病人ID形状: {prob_idx_shape}")
            if age_shape:
                print(f"    * 年龄数据形状: {age_shape}")

# 验证Scaler
print("\n4. 验证Scaler:")
try:
    with open(scaler_path, 'rb') as f:
        loaded_scaler = pickle.load(f)
    
    print(f"  - Scaler加载成功, 特征数量: {len(loaded_scaler.mean_)}")
except Exception as e:
    print(f"  - Scaler验证出错: {str(e)}")

print("\n处理完成！所有数据已按要求按区域标签分别处理并保存，保持了数据结构的完整性。")

In [ ]:
# 单元格10：创建处理汇总信息文件
# 功能：创建数据处理的汇总信息

summary_file = os.path.join(output_base_dir, "processing_summary.txt")

# 统计各数据集的区域和样本数量
train_regions = {}
val_regions = {}
test_regions = {}

for directory, region_dict in [(train_dir, train_regions), (val_dir, val_regions), (test_dir, test_regions)]:
    index_file = os.path.join(directory, "region_index.txt")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            next(f)  # 跳过标题行
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    region_id = int(parts[0])
                    sample_count = int(parts[1])
                    region_dict[region_id] = sample_count

with open(summary_file, 'w') as f:
    f.write("脑区域数据处理汇总信息\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("1. 数据来源\n")
    f.write(f"- 原始数据文件: {data_path}\n\n")
    
    f.write("2. 数据集划分\n")
    f.write(f"- 测试集病人 ({len(patient_allocation['test_patients'])}个): {patient_allocation['test_patients']}\n")
    f.write(f"- 训练和验证集病人 ({len(patient_allocation['train_val_patients'])}个): {patient_allocation['train_val_patients']}\n\n")
    
    f.write("3. 区域统计\n")
    all_regions = sorted(set(list(train_regions.keys()) + list(val_regions.keys()) + list(test_regions.keys())))
    
    f.write(f"- 总区域数: {len(all_regions)}\n")
    f.write("- 各区域样本分布:\n")
    
    for region_id in all_regions:
        train_count = train_regions.get(region_id, 0)
        val_count = val_regions.get(region_id, 0)
        test_count = test_regions.get(region_id, 0)
        total_count = train_count + val_count + test_count
        
        f.write(f"  * 区域 {region_id}: 总计 {total_count} 个样本 (训练: {train_count}, 验证: {val_count}, 测试: {test_count})\n")
    
    f.write("\n4. 数据集统计\n")
    train_total = sum(train_regions.values())
    val_total = sum(val_regions.values())
    test_total = sum(test_regions.values())
    total_samples = train_total + val_total + test_total
    
    f.write(f"- 训练集: {train_total} 个样本, {len(train_regions)} 个区域\n")
    f.write(f"- 验证集: {val_total} 个样本, {len(val_regions)} 个区域\n")
    f.write(f"- 测试集: {test_total} 个样本, {len(test_regions)} 个区域\n")
    f.write(f"- 总样本: {total_samples}\n\n")
    
    f.write("5. 标准化信息\n")
    f.write(f"- Scaler文件: {os.path.basename(scaler_path)}\n")
    f.write(f"- 特征数量: {len(scaler.mean_)}\n\n")
    
    f.write("6. 文件格式\n")
    f.write("- 所有数据以.mat文件格式保存\n")
    f.write("- 每个文件包含完整的原始数据结构: data, region, prob_idx\n")
    f.write("- 如果原始数据有age字段，文件中也包含age字段\n\n")
    
    f.write("7. 处理时间\n")
    f.write(f"- 处理完成时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"处理汇总信息已保存到: {summary_file}")

In [ ]:
# 单元格11：总结和使用说明
print("""
================================================================================
                           数据处理完成总结
================================================================================

处理流程概述:
1. 从TRAIN38.mat文件中加载数据，保留完整的原始结构
2. 将病人分配到测试集和训练+验证集
3. 使用训练和验证集数据拟合StandardScaler并保存
4. 按照102个区域标签分别处理数据，保持数据结构完整性
5. 训练和验证集数据按6:2比例拆分
6. 所有数据保存为.mat文件，保持原始数据的键结构
7. 创建索引文件和辅助函数便于使用

输出文件:
- 训练集数据: {}/train/region_*.mat
- 验证集数据: {}/val/region_*.mat
- 测试集数据: {}/test/region_*.mat
- 测试集按病人分类数据: {}/test/patient_*_region_*.mat
- 数据标准化器: {}/data_scaler.pkl
- 处理汇总信息: {}/processing_summary.txt
- 数据加载辅助函数: {}/data_loader.py

数据文件结构:
每个.mat文件都保留了与原始TRAIN38.mat相同的键结构:
- data: 标准化后的体素特征数据
- region: 102维的区域标签
- prob_idx: 病人ID
- age: 年龄数据（如果原始数据中存在）

使用数据的方法:
1. 使用提供的data_loader.py加载数据:
   - load_region_data(): 加载指定区域的数据
   - load_all_regions(): 加载所有区域的数据
   - load_patient_data(): 加载指定病人的数据
   - load_dataset(): 加载指定数据集（训练、验证或测试）
   - concatenate_regions(): 将多个区域的数据合并

2. 使用保存的Scaler对新数据进行标准化:
   ```python
   from data_loader import load_scaler
   scaler = load_scaler(base_dir)
   new_data_scaled = scaler.transform(new_data)